In [1]:
import warnings
import numpy as np
from typing import Any, List, Union, Dict
import pandas as pd
from IPython.display import display
from pathlib import Path
import os, sys
import itertools 
import time

import napari
from napari.utils.notebook_display import nbscreenshot

from skimage.measure import regionprops_table, label
from skimage.segmentation import watershed

from infer_subc.core.file_io import (read_czi_image,
                                     import_inferred_organelle,
                                     list_image_files)

from infer_subc.core.img import *
from infer_subc.utils.stats import *
from infer_subc.utils.stats import (_assert_uint16_labels)
from infer_subc.utils.stats_helpers import *

from infer_subc.organelles import * 

# from infer_subc.core.file_io import read_czi_image, read_tiff_image
# from infer_subc.core.img import apply_mask
# from infer_subc.utils.batch import list_image_files, find_segmentation_tiff_files
# from infer_subc.utils.stats import surface_area_from_props, get_XY_distribution, get_Z_distribution, _assert_uint16_labels

#For Convexhull Errors
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)

viewer = napari.Viewer()

pd.options.display.max_columns = None

### TODO: need to move this into the code
def _import_inferred_organelle(name: str, suffix: str, meta_dict: Dict, out_data_path: Path, file_type: str) -> Union[np.ndarray, None]:
    """
    read inferred organelle from ome.tif file

    Parameters
    ------------
    name: str
        name of organelle.  i.e. nuc, lyso, etc.
    suffix: str
        the splitter between the file name and segmentation suffix (e.g., "-" if the segmentation name was "01_condition1-mito")
    meta_dict:
        dictionary of meta-data (ome) from original file
    out_data_path:
        Path object of directory where tiffs are read from
    file_type: 
        The type of file you want to import as a string (ex - ".tif", ".tiff", ".czi", etc.)

    Returns
    -------------
    exported file name

    """

    # copy the original file name to meta
    img_name = Path(meta_dict["file_name"])  #
    # add params to metadata
    if name is None:
        pass
    else:
        organelle_fname = f"{img_name.stem}{suffix}{name}{file_type}"

        organelle_path = out_data_path / organelle_fname

        if Path.exists(organelle_path):
            # organelle_obj, _meta_dict = read_ome_image(organelle_path)
            organelle_obj = read_tiff_image(organelle_path)  # .squeeze()
            print(f"loaded  inferred {len(organelle_obj.shape)}D `{name}`  from {out_data_path} ")
            return organelle_obj
        else:
            print(f"`{name}` object not found: {organelle_path}")
            raise FileNotFoundError(f"`{name}` object not found: {organelle_path}")

C:\Users\zscoman\AppData\Local\Temp\ipykernel_27108\2012298414.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


KeyboardInterrupt: 

In [ ]:
def _make_contact_dict(obj_names: list[str],                                        #Intakes list of object names
                       obj_segs: list[np.ndarray]):                                 #Intakes list of object segmentations
    objs_labeled = {}                                                       #Initialize dictionary
    for idx, name in enumerate(obj_names):                                  #Loop across each organelle name
        if name == 'ER':                                                    #Proceed only for ER
            objs_labeled[name]=(obj_segs[idx]>0).astype(np.uint16)          #Ensures ER is labeled only as one object & sets it as key for its object segmentation
        else:                                                               #Proceed for other organelles
            objs_labeled[name]=obj_segs[idx]                                #Set the organelle name as the key for the corresponding object segmentation
    return objs_labeled                                                 #Return a dictionary of segmented objects with keys as the organelle name

In [ ]:
def _create_contact(orgs:str,
                    organelle_segs: dict[str:np.ndarray],
                    splitter: str="X") -> tuple[np.ndarray, np.ndarray]: 
    ##########################################
    ## CREATE CONTACT
    ##########################################
    site = np.ones_like(organelle_segs[orgs.split(splitter)[0]])
    for org in orgs.split(splitter):
        # Creating desired overlap regions
        b = organelle_segs[org]             #collects organelle b
        valid = (b>0)*(site>0)
        digit = len(str(np.max(site)))      #finds number of digits in largest valued label of organelle a segmentation
        site = (b*(10**(digit)))+site       #assigns unique labels to each overlap
        site[valid.astype(bool)==False]=0   #ensures that locations with no overlap are labeled as 0
        site = label(site)                  #simplifies labels
    ##########################################
    ## DETERMINE REDUNDANT CONTACTS
    ##########################################
    LOc_NR = site.copy()
    for org, val in organelle_segs.items():
        if (org not in orgs.split(splitter)) and np.any(site*val):
            print(f"Examining {org} Higher Order contacts", end="\r")
            digit = len(str(np.max(val)))
            valid = (LOc_NR>0)*(val>0)
            HOc = (LOc_NR*(10**(digit)))+val
            HOc[valid.astype(bool)==False]=0
            HOc = label(HOc)
            maxi = len(np.unique(LOc_NR[HOc>0]))
            for num, id in enumerate(np.unique(LOc_NR[HOc > 0])):
                per = round((100*((num+1)/maxi)), 2)
                LOc_NR[LOc_NR==id] = 0
                print(f"Examining {org} Higher Order contacts: {per}% complete", end="\r")
            print(f"Examining {org} Higher Order contacts: {100.00}% complete")
    return site, LOc_NR

In [ ]:
def _get_contact_metrics_3D(list_obj_names: List[str],
                            list_obj_segs: List[np.ndarray],
                            list_region_names: List[str],
                            list_region_segs: List[np.ndarray],
                            mask: np.ndarray,
                            img_f: str,
                            splitter: str="X",
                            scale: Union[tuple, None]=None,
                            include_dist:bool=False, 
                            dist_centering_obj: Union[np.ndarray, None]=None,
                            dist_num_bins: Union[int, None]=None,
                            dist_zernike_degrees: Union[int, None]=None,
                            dist_center_on: Union[bool, None]=None,
                            dist_keep_center_as_bin: Union[bool, None]=None) -> list:
    
    """ 
    Parameters:
    ----------
    list_obj_names: List[str],
        A list of object names to include in the contact analysis
    list_obj_segs: List[np.ndarray],
        A list of numpy arrays associated to each of the names in list_obj_names. The expectation is that these images will be binary segmentation or uint16 labeled images.
    list_region_names: List[str],
        A list of regions to analysis from. Current versions will only utilize:
            - the mask object (identifying the area to analyze from) and
            - a centering object (used to define the center of the XY region for distribution analysis; if none is defined, the default will be the center of the mask region)
    list_region_segs: List[np.ndarray],
        A list of numpy arrays associated to each of the region names. The expectations is that these images will be a binary image. The masking object is expected to be only one object per image.
    mask: np.ndarray,
        The name of the object to use for masking
    splitter: str="_",
        The character you would like to separate object names in the new contact sites names.
        Ex) splitter="_"
            list_obj_names=['ER', 'golgi']
            --> 'ER_golgi'
    scale: Union[tuple, None]=None,
        The dimentions of the image voxels (Z,Y,X). If not scale, the default is (1,1,1)
    include_dist:bool=False, 
        True --> quantify the distribution of contact sites within the masked region
        False --> do not quantify distributions
    dist_centering_obj: Union[np.ndarray, None]=None,
        See get_XY_distribution() for information.
    dist_num_bins: Union[int, None]=None,
        See get_XY_distribution() for information.
    dist_zernike_degrees: Union[int, None]=None,
        See get_XY_distribution() for information.
    dist_center_on: Union[bool, None]=None,
        See get_XY_distribution() for information.
    dist_keep_center_as_bin:
        See get_XY_distribution() for information.

    Output:
    -------
    --> props_table: pd.Dataframe()
        A Pandas dataframe containing quantification of the amount, size, and shape of each contact site.
    dist_tabs: pd.Dataframe() (ONLY IF include_dist=True)
        A Pandas dataframe containing quantification of the contact sites distribution per cell in XY and Z.
    """

    # Properties to measure in regionprops
    properties = ["label", "centroid", "bbox", "area", 
                "equivalent_diameter", "extent", "euler_number", 
                "solidity", "axis_major_length", "slice"]
    
    # preparing mask object
    mask = list_region_segs[list_region_names.index(mask)]

    # preparing centering object
    center_obj = list_region_segs[list_region_names.index(dist_centering_obj)]


    ##########################################
    ## CREATING CONTACT SITE LIST
    ##########################################
    org_dict = _make_contact_dict(list_obj_names, list_obj_segs)

    all_pos =[]
    for n in list(map(lambda x:x+2, (range(len(list_obj_names)-1)))):
        all_pos += itertools.combinations(list_obj_names, n)
    possib = [splitter.join(cont) for cont in all_pos]


    ##########################################
    ## LOOP THROUGH EACH SITE AND QUANTIFY
    ##########################################
    props_tabs = []
    dist_tabs = []
    for cont in possib:
        print(cont)
        site, LOc_NR = _create_contact(cont, org_dict, splitter)
        labels = label(apply_mask(site, mask)).astype(int)
        para_labels = apply_mask((LOc_NR>0), mask).astype(int) * labels
    

        ## quantify amount, size, and shape
        props = regionprops_table(labels, intensity_image=None, properties=properties, extra_properties=None, spacing=scale)
        surface_area_tab = pd.DataFrame(surface_area_from_props(labels, props, scale))


        ## LIST WHICH ORGANELLES ARE INVOLVED IN THE CONTACT
        cont_inv = []
        involved = cont.split(splitter)
        indexes = dict.fromkeys(involved, [])
        indexes[cont] = []

        redundancy = []
        for index, l in enumerate(props["label"]):
            cont_inv.clear()
            present = para_labels[props["slice"][index]]
            present = present==l
            redundant = not np.any(present)
            redundancy.append(redundant)
            for org in involved:
                volume = labels[props["slice"][index]]
                lorg = org_dict[org][props["slice"][index]]
                volume = volume==l
                lorg = lorg[volume]                                 
                all_inv = np.unique(lorg[lorg>0]).tolist()          
                if len(all_inv) != 1:
                    print(f"we have an error.  as-> {all_inv}")
                indexes[org].append(all_inv[0])
                cont_inv.append(f"{all_inv[0]}")
            indexes[cont].append('_'.join(cont_inv))
        

        ## CREATE COMBINED DATAFRAME OF THE QUANTIFICATION
        props_table = pd.DataFrame(props)
        props_table.drop(columns=['slice', 'label'], inplace=True)
        props_table.insert(0, 'label',value=indexes[cont])
        props_table.insert(0, "object", cont)
        props_table.rename(columns={"area": "volume"}, inplace=True)
        props_table.insert(11, "surface_area", surface_area_tab)
        props_table.insert(13, "SA_to_volume_ratio", 
        props_table["surface_area"].div(props_table["volume"]))
        if scale is not None:
            round_scale = (round(scale[0], 4), round(scale[1], 4), round(scale[2], 4))
            props_table.insert(loc=2, column="scale", value=f"{round_scale}")
        else: 
            props_table.insert(loc=2, column="scale", value=f"{tuple(np.ones(labels.ndim))}")
        props_table.insert(2, "in_higher_order", list(map(bool, redundancy)))
        props_tabs.append(props_table)


        ## optional: DISTRIBUTION OF CONTACTS MEASUREMENTS
        if include_dist:
            XY_contact_dist, XY_bins, XY_wedges = get_XY_distribution(mask=mask, 
                                                                      obj=site,
                                                                      obj_name=cont,
                                                                      centering_obj=center_obj,
                                                                      scale=scale,
                                                                      center_on=dist_center_on,
                                                                      keep_center_as_bin=dist_keep_center_as_bin,
                                                                      num_bins=dist_num_bins,
                                                                      zernike_degrees=dist_zernike_degrees)
        
            Z_contact_dist = get_Z_distribution(mask=mask,
                                                obj=site,
                                                obj_name=cont,
                                                center_obj=center_obj,
                                                scale=scale)
            contact_dist_tab = pd.merge(XY_contact_dist, Z_contact_dist, on=["object", "scale"])
            dist_tabs.append(contact_dist_tab)

        indexes.clear()
    

    ## JOIN PER SITE TABLES TO CREATE TWO FINAL TABLES
    props_tables = pd.concat(props_tabs)
    props_tables.insert(loc=0,column='image_name',value=img_f.stem)


    if include_dist:
        dist_tables = pd.concat(dist_tabs)
        dist_tables.insert(loc=0,column='image_name',value=img_f.stem)
        return props_tables, dist_tables
    else:
        return props_tables

In [ ]:
def ncont_batch(out_file_name: str,
                raw_path: str,
                seg_path: str,
                out_path: str,
                mask: str,
                organelle_names: list[str],
                masks_file_name: list[str],
                raw_file_type: str=".tiff",
                scale: bool=True,
                splitter: str="X",
                include_contact_dist: bool=True,
                centering: Union[str, None]=None,
                num_bins: Union[int, None]=None,
                zernike_degrees: Union[int, None]=None,
                center_on: Union[bool, None]= None,
                center_as_bin: Union[bool, None]= None,
                seg_suffix: Union[str, None]="-"):
    start = time.time()
    count = 0

    if isinstance(raw_path, str): raw_path = Path(raw_path)
    if isinstance(seg_path, str): seg_path = Path(seg_path)
    if isinstance(out_path, str): out_path = Path(out_path)

    if not Path.exists(out_path):
        Path.mkdir(out_path)
        print(f"making {out_path}")
    
    # reading list of files from the raw path
    img_file_list = list_image_files(raw_path, raw_file_type)

    # list of segmentation files to collect
    segs_to_collect = organelle_names + masks_file_name

    contact_tabs = []
    distrib_tabs = []
    for img_f in img_file_list:
        print(img_f.stem)
        count = count + 1
        filez = find_segmentation_tiff_files(img_f, segs_to_collect, seg_path, seg_suffix)

        # read in raw file and metadata
        img_data, meta_dict = read_czi_image(filez["raw"])

        # define the scale
        if scale:
            scale_tup = meta_dict['scale']
        else:
            scale_tup = None
        
        regions = [read_tiff_image(filez[region]) for region in masks_file_name]
        organelles = [read_tiff_image(filez[org]) for org in organelle_names]

        ####################################################################################################
        if include_contact_dist:
            for org in (organelle_names):
                XY_org_distribution, XY_bin_masks, XY_wedge_masks = get_XY_distribution(mask=regions[masks_file_name.index(mask)],
                                                                                    centering_obj=regions[masks_file_name.index(centering)],
                                                                                    obj=organelles[organelle_names.index(org)],
                                                                                    obj_name=org,
                                                                                    scale=scale_tup,
                                                                                    num_bins=num_bins,
                                                                                    center_on=center_on,
                                                                                    keep_center_as_bin=center_as_bin,
                                                                                    zernike_degrees=zernike_degrees)
                Z_org_distribution = get_Z_distribution(mask=regions[masks_file_name.index(mask)], 
                                                        obj=organelles[organelle_names.index(org)],
                                                        obj_name=org,
                                                        center_obj=regions[masks_file_name.index(centering)],
                                                        scale=scale_tup)
            
                org_distribution_metrics = pd.merge(XY_org_distribution, Z_org_distribution,on=["object", "scale"])

                distrib_tabs.append(org_distribution_metrics)
        
        #####################################################################################################
        if len(organelle_names) >= 2:
            if include_contact_dist:
                cont_tabs, dist_tabs = _get_contact_metrics_3D(list_obj_names = organelle_names,
                                                               list_obj_segs = organelles,
                                                               list_region_names = masks_file_name,
                                                               list_region_segs= regions,
                                                               mask = mask,
                                                               img_f = img_f,
                                                               splitter = splitter,
                                                               scale = scale_tup,
                                                               include_dist = include_contact_dist, 
                                                               dist_centering_obj = centering,
                                                               dist_num_bins = num_bins,
                                                               dist_zernike_degrees = zernike_degrees,
                                                               dist_center_on = center_on,
                                                               dist_keep_center_as_bin = center_as_bin)
            else:
                cont_tabs = _get_contact_metrics_3D(list_obj_names = organelle_names,
                                                    list_obj_segs = organelles,
                                                    list_region_names = masks_file_name,
                                                    list_region_segs= regions,
                                                    mask = mask,
                                                    img_f = img_f,
                                                    splitter = splitter,
                                                    scale = scale_tup,
                                                    include_dist = include_contact_dist, 
                                                    dist_centering_obj = centering,
                                                    dist_num_bins = num_bins,
                                                    dist_zernike_degrees = zernike_degrees,
                                                    dist_center_on = center_on,
                                                    dist_keep_center_as_bin = center_as_bin)
        #####################################################################################################
        contact_tabs.append(cont_tabs)
        cont_tabs.head()
        if include_contact_dist:
           distrib_tabs.append(dist_tabs)
        end2 = time.time()
        print(f"Completed processing for {count} images in {(end2-start)/60} mins.")


    final_contact = pd.concat(contact_tabs, ignore_index=True)
    contact_csv_path = out_path / f"{out_file_name}contacts.csv"
    final_contact.to_csv(contact_csv_path)

    if include_contact_dist:
        final_dist = pd.concat(distrib_tabs, ignore_index=True)
        dist_csv_path = out_path / f"{out_file_name}distributions.csv"
        final_dist.to_csv(dist_csv_path)
    
    end = time.time()
    print(f"Quantification for {count} files is COMPLETE! Files saved to '{out_path}'.")
    print(f"It took {(end - start)/60} minutes to quantify these files.")
    return cont_tabs

In [ ]:
ncont_batch(out_file_name ="declumpeddata101824",
            raw_path ="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/raw_single",
            seg_path ="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/out_single",
            out_path ="C:/Users/zscoman/Documents/Python Scripts/Infer-subc-2D/quant_single",
            mask = "cell",
            organelle_names = ["ER","golgi", "LD", "lyso", "mito", "nuc", "perox"],
            masks_file_name = ["cell","nuc"],
            raw_file_type = ".tiff",
            scale = True,
            splitter = "X",
            include_contact_dist = False,
            centering = "nuc",
            num_bins = 5,
            zernike_degrees = None,
            center_on = False,
            center_as_bin = True).head()